# 목표설정이 학업성취에 미치는 영향 분석
### OECD PISA 2018 데이터 기반 - 목표 지향 유형별 수학 성취도 예측

**분석 질문**: 학생의 목표 설정 방식(숙달목표 vs 실패 회피)이 수학 성취도를 얼마나 예측하는가?

## 0. 환경 설정

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, roc_auc_score, silhouette_score
from xgboost import XGBClassifier

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)

print('라이브러리 로드 완료')

라이브러리 로드 완료


## 1. 데이터 로드 및 변수 추출

In [2]:
# PISA 2018 데이터 로드
# 다운로드: https://www.oecd.org/pisa/data/2018database/
# CY07_MSU_STU_QQQ.sav (SPSS 형식) → data/ 폴더에 저장

# pyreadstat 설치 필요: pip install pyreadstat
import pyreadstat

df_full, meta = pyreadstat.read_sav('../data/CY07_MSU_STU_QQQ.sav',
                                     usecols=['CNT', 'CNTSTUID',
                                              'MASTGOAL', 'GFOFAIL', 'SWBP', 'BELONG',
                                              'PV1MATH', 'PV2MATH', 'PV3MATH',
                                              'PV4MATH', 'PV5MATH'])
print(f'전체 데이터: {df_full.shape}')
df_full.head()

PyreadstatError: File ../data/CY07_MSU_STU_QQQ.sav does not exist!

In [ ]:
# 수학 성취도: 5개 추정값 평균 사용
math_cols = ['PV1MATH', 'PV2MATH', 'PV3MATH', 'PV4MATH', 'PV5MATH']
df_full['math_score'] = df_full[math_cols].mean(axis=1)

# 분석 대상 변수
goal_vars = ['MASTGOAL', 'GFOFAIL', 'SWBP', 'BELONG']
df = df_full[goal_vars + ['math_score', 'CNT']].dropna()

# 타겟: 수학 성취도 상위 50% = 1 (성취), 하위 50% = 0 (저성취)
df['high_achieve'] = (df['math_score'] >= df['math_score'].median()).astype(int)

print(f'분석 데이터: {df.shape[0]:,}명')
print(f'수학 점수 평균: {df["math_score"].mean():.1f}, 중앙값: {df["math_score"].median():.1f}')
print(f'\n[변수 설명]')
print('MASTGOAL: 숙달목표 지향성 (높을수록 배움 자체를 목표로 함)')
print('GFOFAIL: 실패에 대한 두려움 (높을수록 실패를 두려워함)')
print('SWBP: 주관적 웰빙 (높을수록 학교생활 만족)')
print('BELONG: 학교 소속감 (높을수록 학교에 소속감 느낌)')

## 2. 탐색적 데이터 분석 (EDA)

In [ ]:
# 목표 변수별 분포 (고성취 vs 저성취 비교)
var_labels = {
    'MASTGOAL': '숙달목표 지향성',
    'GFOFAIL': '실패 두려움',
    'SWBP': '주관적 웰빙',
    'BELONG': '학교 소속감'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(goal_vars):
    low = df[df['high_achieve'] == 0][var]
    high = df[df['high_achieve'] == 1][var]
    axes[i].hist(low, bins=30, alpha=0.6, color='#e74c3c', label='저성취', density=True)
    axes[i].hist(high, bins=30, alpha=0.6, color='#2ecc71', label='고성취', density=True)
    axes[i].set_title(var_labels[var], fontsize=13, fontweight='bold')
    axes[i].legend(fontsize=11)
    axes[i].set_xlabel('점수')
    axes[i].set_ylabel('밀도')

plt.suptitle('목표설정 변수별 고성취/저성취 분포 비교', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/goal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 목표 변수 × 수학 점수 상관관계 히트맵
corr_df = df[goal_vars + ['math_score']].copy()
corr_df.columns = [var_labels.get(c, c) for c in corr_df.columns]

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, mask=mask, square=True, linewidths=1,
            annot_kws={'size': 11})
plt.title('목표설정 변수 × 수학 성취도 상관관계', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n[핵심 상관계수]')
for var in goal_vars:
    corr = df[var].corr(df['math_score'])
    direction = '↑ 긍정' if corr > 0 else '↓ 부정'
    print(f'{var_labels[var]:15s}: r = {corr:+.3f}  {direction}')

In [ ]:
# 숙달목표 분위별 수학 점수 박스플롯
df['mastgoal_q'] = pd.qcut(df['MASTGOAL'], q=4,
                            labels=['하위 25%\n(목표 없음)', '25-50%', '50-75%', '상위 25%\n(목표 뚜렷)'])

plt.figure(figsize=(10, 6))
box_data = [df[df['mastgoal_q'] == q]['math_score'].values
            for q in df['mastgoal_q'].cat.categories]
bp = plt.boxplot(box_data, patch_artist=True,
                 labels=df['mastgoal_q'].cat.categories)
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

means = [np.mean(d) for d in box_data]
for i, mean in enumerate(means):
    plt.text(i+1, mean + 5, f'{mean:.0f}점', ha='center', fontsize=10, fontweight='bold')

plt.title('숙달목표 지향성 수준별 수학 성취도', fontsize=14, fontweight='bold')
plt.ylabel('수학 점수')
plt.xlabel('숙달목표 지향성 분위')
plt.tight_layout()
plt.savefig('../data/mastgoal_vs_math.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 목표 유형 군집 분석 (K-Means)

In [3]:
# K-Means로 목표 유형 군집화
X_cluster = df[goal_vars].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# 최적 k 탐색 (Elbow + Silhouette)
inertias, silhouettes = [], []
k_range = range(2, 7)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].set_xlabel('k (군집 수)')
axes[0].set_ylabel('Inertia')

axes[1].plot(k_range, silhouettes, 'ro-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_xlabel('k (군집 수)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(silhouettes)]
print(f'최적 군집 수: k = {best_k}')

NameError: name 'df' is not defined

In [4]:
# 최적 k로 군집 분석
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

# 군집별 특성 분석
cluster_profile = df.groupby('cluster')[goal_vars + ['math_score']].mean()
cluster_profile.columns = [var_labels.get(c, c) for c in cluster_profile.columns]

# 군집에 이름 붙이기 (숙달목표 기준)
cluster_profile['군집명'] = ''
mastgoal_col = var_labels['MASTGOAL']
sorted_clusters = cluster_profile[mastgoal_col].sort_values()

cluster_names = {}
for rank, idx in enumerate(sorted_clusters.index):
    if rank == 0: cluster_names[idx] = '목표 무관심형'
    elif rank == len(sorted_clusters) - 1: cluster_names[idx] = '목표 주도형'
    else: cluster_names[idx] = f'목표 모색형 {rank}'

df['cluster_name'] = df['cluster'].map(cluster_names)

print('=== 군집별 평균 프로파일 ===')
display_df = cluster_profile.drop(columns='군집명')
display_df.index = [cluster_names[i] for i in display_df.index]
print(display_df.round(3))

NameError: name 'best_k' is not defined

In [5]:
# 군집별 수학 성취도 비교 시각화
cluster_math = df.groupby('cluster_name')['math_score'].agg(['mean', 'std', 'count'])
cluster_math = cluster_math.sort_values('mean', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 평균 점수 비교
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(cluster_math)))
bars = axes[0].barh(cluster_math.index, cluster_math['mean'],
                    xerr=cluster_math['std']/np.sqrt(cluster_math['count']),
                    color=colors, alpha=0.85, capsize=4)
axes[0].set_title('목표 유형별 수학 성취도 평균', fontsize=13, fontweight='bold')
axes[0].set_xlabel('수학 점수')
for bar, val in zip(bars, cluster_math['mean']):
    axes[0].text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}점', va='center', fontsize=10)

# 레이더 차트 (군집 프로파일)
categories = list(var_labels.values())
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

ax2 = plt.subplot(122, polar=True)
radar_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for i, (cluster_id, name) in enumerate(cluster_names.items()):
    vals = [cluster_profile.loc[cluster_id, var_labels[v]] for v in goal_vars]
    vals_norm = [(v - df[goal_vars[j]].min()) / (df[goal_vars[j]].max() - df[goal_vars[j]].min())
                 for j, v in enumerate(vals)]
    vals_norm += vals_norm[:1]
    ax2.plot(angles, vals_norm, 'o-', linewidth=2,
             color=radar_colors[i % len(radar_colors)], label=name)
    ax2.fill(angles, vals_norm, alpha=0.1, color=radar_colors[i % len(radar_colors)])

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories, fontsize=9)
ax2.set_title('목표 유형 프로파일\n(레이더 차트)', fontsize=12, fontweight='bold', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig('../data/cluster_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'df' is not defined

## 4. 성취도 예측 모델

In [6]:
# 모델 학습
X = df[goal_vars].values
y = df['high_achieve'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'Logistic Regression': (LogisticRegression(random_state=42, max_iter=1000), True),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'XGBoost': (XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'), False)
}

results = {}
for name, (model, use_scaled) in models.items():
    Xtr = X_train_s if use_scaled else X_train
    Xte = X_test_s if use_scaled else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]
    results[name] = {
        'model': model,
        'accuracy': (y_pred == y_test).mean(),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'y_pred': y_pred
    }
    print(f'{name:25s}: 정확도={results[name]["accuracy"]:.3f}, ROC-AUC={results[name]["roc_auc"]:.3f}')

NameError: name 'df' is not defined

In [7]:
# 최고 성능 모델 Feature Importance
best_name = max(results, key=lambda m: results[m]['roc_auc'])
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
else:
    importances = np.abs(best_model.coef_[0])

fi_df = pd.DataFrame({
    'variable': [var_labels[v] for v in goal_vars],
    'importance': importances
}).sort_values('importance', ascending=True)

plt.figure(figsize=(9, 4))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi_df)))
bars = plt.barh(fi_df['variable'], fi_df['importance'], color=colors, alpha=0.85)
plt.title(f'성취도 예측 변수 중요도 ({best_name})', fontsize=13, fontweight='bold')
plt.xlabel('중요도')
for bar, val in zip(bars, fi_df['importance']):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'results' is not defined

## 5. 결론 및 교육 현장 시사점

In [8]:
print('=' * 55)
print(f'  최고 성능 모델: {best_name}')
print(f'  정확도: {results[best_name]["accuracy"]:.3f}  |  ROC-AUC: {results[best_name]["roc_auc"]:.3f}')
print('=' * 55)
print()
print('[분류 리포트]')
print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['저성취', '고성취']))

print('[핵심 발견]')
top_var = fi_df.iloc[-1]['variable']
print(f'1. 성취도 예측에 가장 중요한 변수: {top_var}')
print(f'2. 목표설정 변수만으로도 성취도를 {results[best_name]["accuracy"]:.1%} 정확도로 예측 가능')
print()
print('[코딩 교육 현장 적용 방안]')
print('- 수업 시작 전 학생의 목표 유형 간단 설문 실시')
print('- "목표 무관심형" 학생에게 단계별 소목표 설정 코칭 제공')
print('- 목표 달성 시 즉각적 피드백으로 숙달목표 지향성 강화')
print('- 실패 두려움이 높은 학생은 오류 친화적 환경 조성 우선')

NameError: name 'best_name' is not defined